# Lab 20: Hilbert Spaces and Applications

This independent-study notebook accompanies Chapter 20.

We use finite computations to understand infinite-dimensional ideas:

- projections using Gram matrices;
- best polynomial approximation in $L^2[0,1]$;
- $\ell^2$ sequence tests;
- Fourier coefficients as Hilbert-space coordinates;
- positive semidefinite kernel Gram matrices.

Each section includes worked solutions and a similar practice question.

## 1. Projection using a Gram matrix

Let

$$
u_1=(1,1,0),\qquad u_2=(1,0,1),\qquad y=(2,1,3).
$$

We want the projection of $y$ onto

$$
\mathcal{M}=\operatorname{span}\{u_1,u_2\}.
$$

If $A=[u_1\ u_2]$, then

$$
\operatorname{Proj}_{\mathcal{M}}y=A(A^TA)^{-1}A^Ty.
$$

In [ ]:
import numpy as np

u1 = np.array([1.0, 1.0, 0.0])
u2 = np.array([1.0, 0.0, 1.0])
y = np.array([2.0, 1.0, 3.0])

A = np.column_stack([u1, u2])

G = A.T @ A
b = A.T @ y
c = np.linalg.solve(G, b)
p = A @ c
r = y - p

print("Gram matrix G:")
print(G)
print("Right-hand side b:", b)
print("Coefficients c:", c)
print("Projection p:", p)
print("Residual r=y-p:", r)
print("A^T r:", A.T @ r)

### Interpretation

The vector `A.T @ r` is numerically zero. This verifies

$$
\langle y-p,u_1\rangle=0,\qquad
\langle y-p,u_2\rangle=0.
$$

So the residual is orthogonal to the subspace.

### Similar practice 1

Change

$$
y=(1,4,2)
$$

and compute the projection onto the same subspace.

In [ ]:
# Similar practice 1: solution

y2 = np.array([1.0, 4.0, 2.0])

b2 = A.T @ y2
c2 = np.linalg.solve(G, b2)
p2 = A @ c2
r2 = y2 - p2

print("New y:", y2)
print("Coefficients:", c2)
print("Projection:", p2)
print("Residual:", r2)
print("Orthogonality check A^T r:", A.T @ r2)

## 2. Best polynomial approximation in $L^2[0,1]$

We approximate $e^x$ by a polynomial of degree at most $d$.

For basis functions

$$
1,x,x^2,\ldots,x^d,
$$

the normal equations are

$$
G\mathbf{c}=\mathbf{b},
$$

where

$$
G_{ij}=\int_0^1 x^{i+j}\,dx,\qquad
b_i=\int_0^1 e^x x^i\,dx.
$$

In [ ]:
import sympy as sp

x = sp.symbols("x")

def best_poly_exp(degree):
    basis = [x**k for k in range(degree+1)]
    f = sp.exp(x)
    G = sp.Matrix([[sp.integrate(basis[j]*basis[i], (x, 0, 1))
                    for j in range(degree+1)] for i in range(degree+1)])
    rhs = sp.Matrix([sp.integrate(f*basis[i], (x, 0, 1)) for i in range(degree+1)])
    c = G.LUsolve(rhs)
    p = sp.expand(sum(c[i]*basis[i] for i in range(degree+1)))
    return G, rhs, c, p

G2, rhs2, c2, p2 = best_poly_exp(2)
print("Gram matrix:")
sp.pretty_print(G2)
print("Coefficients:")
sp.pretty_print(c2)
print("Best quadratic:")
sp.pretty_print(p2)

In [ ]:
# Numerical version and plot

import numpy as np
import matplotlib.pyplot as plt

p2_fun = sp.lambdify(x, p2, "numpy")
xs = np.linspace(0, 1, 400)

plt.figure(figsize=(7,4))
plt.plot(xs, np.exp(xs), label="$e^x$")
plt.plot(xs, p2_fun(xs), label="best quadratic")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Best $L^2[0,1]$ quadratic approximation to $e^x$")
plt.legend()
plt.show()

### Similar practice 2

Find the best linear approximation to $e^x$ on $[0,1]$.

In [ ]:
# Similar practice 2: solution

G1, rhs1, c1, p1 = best_poly_exp(1)
print("Best linear coefficients:")
sp.pretty_print(c1)
print("Best linear polynomial:")
sp.pretty_print(p1)

p1_fun = sp.lambdify(x, p1, "numpy")
plt.figure(figsize=(7,4))
plt.plot(xs, np.exp(xs), label="$e^x$")
plt.plot(xs, p1_fun(xs), label="best linear")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Best $L^2[0,1]$ linear approximation to $e^x$")
plt.legend()
plt.show()

## 3. $\ell^2$ sequence tests

A sequence $a=(a_n)$ belongs to $\ell^2$ if

$$
\sum_{n=1}^{\infty}|a_n|^2<\infty.
$$

We compare three sequences:

$$
a_n=\frac1n,\qquad
b_n=\frac1{\sqrt n},\qquad
c_n=\frac1{2^n}.
$$

In [ ]:
def partial_sums(seq, N=10000):
    n = np.arange(1, N+1)
    if seq == "1/n":
        a = 1/n
    elif seq == "1/sqrt(n)":
        a = 1/np.sqrt(n)
    elif seq == "1/2^n":
        a = 1/(2.0**n)
    return np.cumsum(a*a)

N = 1000
for seq in ["1/n", "1/sqrt(n)", "1/2^n"]:
    s = partial_sums(seq, N)
    print(seq, "partial sum at N =", N, "is", s[-1])

In [ ]:
plt.figure(figsize=(7,4))
for seq in ["1/n", "1/sqrt(n)", "1/2^n"]:
    s = partial_sums(seq, 1000)
    plt.plot(s, label=seq)

plt.xlabel("N")
plt.ylabel(r"$\sum_{n=1}^N |a_n|^2$")
plt.title(r"Partial sums for $\ell^2$ test")
plt.legend()
plt.show()

### Conclusion

- $(1/n)\in \ell^2$, because $\sum 1/n^2$ converges.
- $(1/\sqrt n)\notin \ell^2$, because $\sum 1/n$ diverges.
- $(1/2^n)\in \ell^2$, because it gives a geometric series after squaring.

### Similar practice 3

Decide whether

$$
a_n=\frac{1}{n^{3/4}}
$$

belongs to $\ell^2$.

In [ ]:
# Similar practice 3: solution

# Squaring gives 1/n^(3/2), and p-series with p=3/2 converges.
# Therefore the sequence belongs to ell^2.

N = 10000
n = np.arange(1, N+1)
s = np.cumsum((1/n**(3/4))**2)
print("Partial sum at N =", N, "is", s[-1])
print("Conclusion: 1/n^(3/4) is in ell^2 because sum 1/n^(3/2) converges.")

## 4. Fourier partial sums as projections

For the square wave

$$
f(x)=\operatorname{sign}(\sin x),
$$

the sine Fourier series is

$$
f(x)\sim
\frac4\pi
\left(
\sin x+\frac{\sin 3x}{3}+\frac{\sin 5x}{5}+\cdots
\right).
$$

Truncating this expansion is projection onto a finite-dimensional subspace of low-frequency sine functions.

In [ ]:
xs = np.linspace(-np.pi, np.pi, 1200)

def square_wave(xs):
    return np.sign(np.sin(xs))

def square_partial_sum(xs, num_odd_terms):
    s = np.zeros_like(xs)
    for j in range(num_odd_terms):
        k = 2*j + 1
        s += (4/np.pi) * np.sin(k*xs)/k
    return s

plt.figure(figsize=(8,4))
plt.plot(xs, square_wave(xs), "--", label="square wave")
for terms in [1, 3, 7, 15]:
    plt.plot(xs, square_partial_sum(xs, terms), label=f"{terms} odd terms")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Fourier partial sums")
plt.legend()
plt.show()

### Similar practice 4

Plot the partial sums using 5, 10, and 20 odd terms. Describe what improves and what artifact remains near the jump discontinuities.

In [ ]:
# Similar practice 4: solution

plt.figure(figsize=(8,4))
plt.plot(xs, square_wave(xs), "--", label="square wave")
for terms in [5, 10, 20]:
    plt.plot(xs, square_partial_sum(xs, terms), label=f"{terms} odd terms")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Fourier partial sums: more terms")
plt.legend()
plt.show()

print("Observation: Away from the jumps, the approximation improves.")
print("Near the jumps, oscillation remains; this is the Gibbs phenomenon.")

## 5. Positive semidefinite kernel Gram matrices

A kernel $K$ is positive semidefinite if every Gram matrix

$$
G_{ij}=K(x_i,x_j)
$$

is positive semidefinite.

We use the polynomial kernel

$$
K(x,y)=(1+xy)^2.
$$

In [ ]:
def poly_kernel(x, y):
    return (1 + x*y)**2

points = np.array([-1.0, 0.0, 2.0])
G = np.array([[poly_kernel(xi, xj) for xj in points] for xi in points])

eigvals = np.linalg.eigvalsh(G)

print("Points:", points)
print("Gram matrix:")
print(G)
print("Eigenvalues:", eigvals)
print("Is positive semidefinite?", np.all(eigvals >= -1e-10))

### Similar practice 5

Use points $-2,1,3$ and check the eigenvalues of the Gram matrix.

In [ ]:
# Similar practice 5: solution

points2 = np.array([-2.0, 1.0, 3.0])
G2 = np.array([[poly_kernel(xi, xj) for xj in points2] for xi in points2])
eigvals2 = np.linalg.eigvalsh(G2)

print("Points:", points2)
print("Gram matrix:")
print(G2)
print("Eigenvalues:", eigvals2)
print("Is positive semidefinite?", np.all(eigvals2 >= -1e-10))

## 6. Reflection questions

Answer these in your own words.

1. Why is completeness important for projection?
2. Why are Fourier coefficients like coordinates?
3. Why do we identify functions that agree almost everywhere in $L^2$?
4. How is a kernel Gram matrix related to hidden inner products?

### Suggested answers

1. Completeness ensures that minimizing sequences have limits inside the space.
2. Fourier coefficients are inner products with orthonormal basis vectors.
3. The $L^2$ norm cannot detect changes on sets of measure zero.
4. A positive semidefinite kernel behaves as an inner product after mapping data into a feature space.